**Autor:** Alan Sastre  
**GitHub:** [github.com/alansastre/genai-agentes](https://github.com/alansastre/genai-agentes)

---


# Middleware integrados en LangChain

En LangChain los agentes creados con `create_agent` admiten **middleware**: componentes que se ejecutan antes o después de cada paso del bucle del agente (llamada al modelo, selección de herramientas, ejecución de herramientas). Permiten controlar costes, cumplimiento, resiliencia y comportamiento sin modificar la lógica del agente.

A continuación se explican **qué son**, **cómo se integran** (un solo patrón: la lista `middleware` en `create_agent`) y **varios middleware preconstruidos**, en orden de complejidad. Cada ejemplo está pensado para que el middleware se ejecute y se pueda ver su efecto.

## Qué son los middleware y para qué sirven

Los middleware permiten:

- **Limitar uso:** límite de llamadas al modelo o a herramientas (coste y bucles infinitos).
- **Cumplimiento y seguridad:** detección y ofuscación de datos personales (PII), aprobación humana antes de ejecutar ciertas herramientas.
- **Resiliencia:** reintentos con backoff y fallback a otro modelo si el principal falla.
- **Contexto largo:** resumir el historial cuando se acerca al límite de tokens y mantener solo los mensajes recientes.

Se añaden al crear el agente pasando una lista al parámetro `middleware` de `create_agent`. El orden puede influir en el comportamiento, ya que se encadenan según su posición.

> Los middleware se encadenan en el orden en que aparecen en la lista; conviene tenerlo en cuenta al combinar varios.

## El bucle del agente y los hooks

El bucle básico del agente consiste en: (1) llamar al modelo, (2) decidir si usar herramientas, (3) ejecutar las herramientas elegidas y (4) repetir hasta que el modelo no invoque más herramientas y devuelva la respuesta final.

Los middleware exponen **hooks** antes y después de cada uno de esos pasos. Así puedes interceptar y modificar peticiones al modelo, resultados de herramientas o el flujo de mensajes (por ejemplo inyectando un resumen del historial o aplicando límites).

Sin hooks, el bucle típico de un agente es:

```mermaid
flowchart TB
    req([request])
    mdl[model]
    tls[tools]
    res([result])
    req --> mdl
    mdl -. action .-> tls
    tls -- observation --> mdl
    mdl -.-> res
```

Con middleware, el mismo bucle queda envuelto por hooks:

```mermaid
flowchart TB
    req([request])
    res([result])
    ba[before_agent]
    bm[before_model]
    wtc[wrap_tool_call]
    wmc[wrap_model_call]
    mdl[model]
    tls[tools]
    am[after_model]
    aa[after_agent]
    req --> ba
    ba --> bm
    bm --> wmc
    wmc --> mdl
    mdl --> am
    am -.-> wtc
    wtc --> tls
    tls --> bm
    am -.-> aa
    aa --> res
```

Las flechas punteadas representan rutas condicionales: en el diagrama base, `model` puede terminar en `result` o invocar `tools`; en el diagrama con hooks, tras `after_model` el flujo puede volver al bucle de herramientas (`after_model` → `wrap_tool_call` → `tools` → `before_model`) o avanzar al cierre (`after_model` → `after_agent`).

No es necesario implementar esos hooks para usar los middleware integrados: basta con instanciarlos y añadirlos a la lista `middleware`. A continuación se ven varios de ellos, del más simple al que requiere más contexto.

> Con los integrados no hace falta tocar los hooks; solo instanciar el middleware y pasarlo en `create_agent`.

---

## 1. PIIMiddleware: redactar datos personales

**Objetivo:** que el modelo nunca vea datos sensibles en claro. El middleware detecta PII (por ejemplo emails) en la entrada y los sustituye por una etiqueta (`[REDACTED_email]`) antes de que lleguen al modelo. Así la respuesta no puede filtrar el dato real.

Es el ejemplo más directo para ver el **patrón** de uso: `create_agent(..., middleware=[...])` y una sola invocación donde el middleware actúa.

> Con `apply_to_input=True` el modelo nunca recibe el dato sensible; solo la etiqueta de redacción.

In [1]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[
        PIIMiddleware("email", strategy="redact", apply_to_input=True),
    ],
)
r = agent.invoke({
    "messages": [HumanMessage(content="Di OK y repite mi email: soporte@empresa.com")],
})
last = r["messages"][-1]
print("Respuesta:", last.content[:250] if last.content else "")
print("\nEl middleware redactó el email; el modelo solo vio [REDACTED_email], por eso la respuesta no contiene el correo real.")

Respuesta: OK [REDACTED_EMAIL]

El middleware redactó el email; el modelo solo vio [REDACTED_email], por eso la respuesta no contiene el correo real.


Otras estrategias: `mask`, `hash`, `block`. Se puede aplicar también a la salida del modelo o a resultados de herramientas (`apply_to_output`, `apply_to_tool_results`).

---

## 2. SummarizationMiddleware: comprimir historial largo

**Objetivo:** evitar desbordar la ventana de contexto. Cuando el historial cumple una condición (p. ej. número de mensajes o de tokens), el middleware genera un resumen y sustituye mensajes antiguos por ese resumen, manteniendo solo los últimos N mensajes sin resumir.

Parámetros clave:

- **trigger:** cuándo resumir: `("tokens", N)`, `("messages", N)` o `("fraction", 0.0 a 1.0)` del contexto.
- **keep:** cuánto conservar tras el resumen (mismo tipo de tuplas).
- **model:** modelo usado solo para generar el resumen (puede ser más barato).

En el ejemplo se usa `trigger=("messages", 4)` y `keep=("messages", 2)` para que el resumen se dispare en el **tercer turno** (cuando ya hay 4 mensajes). Así se ve cómo el historial se comprime y el agente sigue respondiendo usando el resumen.

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[
        SummarizationMiddleware(
            model="gpt-5-nano",
            trigger=("messages", 4),
            keep=("messages", 2),
        ),
    ],
)
# Turno 1: 1 mensaje de usuario; tras respuesta quedan 2 mensajes
messages = [HumanMessage(content="Dime en una frase qué es Python.")]
r1 = agent.invoke({"messages": messages})
messages = list(r1["messages"])
print("Turno 1 - Mensajes en estado:", len(messages))
print("Última respuesta:", messages[-1].content[:100] + "...")

Turno 1 - Mensajes en estado: 2
Última respuesta: Python es un lenguaje de programación de alto nivel, interpretado y multiparadigma, con sintaxis cla...


In [3]:
# Turno 2: añadimos otro usuario (3 mensajes); el modelo responde (4 mensajes)
messages.append(HumanMessage(content="¿Y en qué se diferencia de Java en una frase?"))
r2 = agent.invoke({"messages": messages})
messages = list(r2["messages"])
print("Turno 2 - Mensajes en estado:", len(messages))

Turno 2 - Mensajes en estado: 4


In [4]:
# Turno 3: con 4 mensajes se dispara el resumen; el agente ve resumen + 2 recientes
messages.append(HumanMessage(content="¿Qué lenguaje pregunté primero?"))
r3 = agent.invoke({"messages": messages})
messages = list(r3["messages"])
print("Turno 3 (tras resumen) - Mensajes en estado:", len(messages))
for i, m in enumerate(messages[:3]):
    preview = (m.content or "")[:70].replace("\n", " ")
    print(f"  {i+1}: {preview}...")
print("\nÚltima respuesta:", messages[-1].content[:150])

Turno 3 (tras resumen) - Mensajes en estado: 4
  1: Here is a summary of the conversation to date:  Python is a high-level...
  2: Python es un lenguaje interpretado, de tipado dinámico y sintaxis conc...
  3: ¿Qué lenguaje pregunté primero?...

Última respuesta: Preguntaste primero por Python.


En el turno 3 el primer mensaje es el resumen generado por el middleware; el agente puede responder "Python" porque lo leyó en ese resumen.

---

## 3. ModelCallLimitMiddleware: límite de llamadas al modelo

**Objetivo:** acotar coste y bucles. Limita cuántas veces se llama al modelo por ejecución (`run_limit`) o por hilo de conversación (`thread_limit`). Para que el límite por hilo persista entre invocaciones hace falta un **checkpointer** (memoria a corto plazo); aquí se usa `InMemorySaver`.

La utilidad real se entiende comparando límites: en un flujo con herramientas, `run_limit=1` suele ser demasiado estricto (el agente decide herramienta pero no llega a redactar la respuesta final), mientras que `run_limit=2` permite completar ese caso normal y sigue evitando iteraciones largas.

In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain_core.messages import HumanMessage

@tool
def obtener_dato() -> str:
    """Devuelve un dato simple para probar el límite de llamadas al modelo."""
    return "5"

def ejecutar_con_limite(run_limit: int):
    agent = create_agent(
        model="gpt-5-mini",
        tools=[obtener_dato],
        checkpointer=InMemorySaver(),
        middleware=[
            ModelCallLimitMiddleware(thread_limit=10, run_limit=run_limit, exit_behavior="end"),
        ],
    )
    r = agent.invoke(
        {
            "messages": [
                HumanMessage(
                    content="No respondas directamente. Usa la herramienta obtener_dato y luego responde exactamente: El número es <dato>."
                )
            ]
        },
        config={"configurable": {"thread_id": f"limit-demo-{run_limit}"}},
    )
    msgs = r["messages"]
    tool_msgs = [m for m in msgs if m.__class__.__name__ == "ToolMessage"]
    ai_msgs = [m for m in msgs if m.__class__.__name__ == "AIMessage"]
    last_ai = ai_msgs[-1].content if ai_msgs else None

    print(f"run_limit={run_limit}")
    print("  ToolMessage detectados:", len(tool_msgs))
    print("  Último AIMessage:", repr(last_ai))
    print()

ejecutar_con_limite(1)
ejecutar_con_limite(2)
print("Con run_limit=2 el agente suele completar 'El número es 5'; con run_limit=1 se corta antes para evitar sobrecoste/bucles.")

run_limit=1
  ToolMessage detectados: 1
  Último AIMessage: 'Model call limits exceeded: run limit (1/1)'

run_limit=2
  ToolMessage detectados: 1
  Último AIMessage: 'El número es 5.'

Con run_limit=2 el agente suele completar 'El número es 5'; con run_limit=1 se corta antes para evitar sobrecoste/bucles.


`exit_behavior="end"` termina la ejecución de forma controlada; con `"error"` se lanzaría una excepción.

---

## 4. ToolCallLimitMiddleware: límite de llamadas a herramientas

**Objetivo:** limitar cuántas veces se ejecutan las herramientas (por run, por thread o por herramienta concreta con `tool_name`). Frecuente en APIs externas o búsquedas.

Con `thread_limit=2` y `exit_behavior="continue"` puedes consumir dos tool calls en una primera ejecución y bloquear llamadas extra en la siguiente, manteniendo una respuesta final.

In [10]:
from langchain.agents import create_agent
from langchain.agents.middleware import ToolCallLimitMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.tools import tool
from langchain_core.messages import HumanMessage

real_calls = {"buscar": 0, "consultas": []}

@tool
def buscar(consulta: str) -> str:
    """Busca información sobre la consulta. Devuelve un resultado simulado."""
    real_calls["buscar"] += 1
    real_calls["consultas"].append(consulta)
    return f"Resultado para: {consulta}"

agent = create_agent(
    model="gpt-5-mini",
    tools=[buscar],
    checkpointer=InMemorySaver(),
    middleware=[
        ToolCallLimitMiddleware(thread_limit=2, exit_behavior="continue"),
    ],
)
config = {"configurable": {"thread_id": "tool-limit-demo"}}

# Run 1: consume las 2 llamadas permitidas del hilo.
r1 = agent.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Haz exactamente dos búsquedas separadas con la herramienta buscar: "
                    "'LangChain' y 'middleware'. Después responde en una frase con ambos resultados."
                )
            )
        ]
    },
    config=config,
)
print("Run 1 - consultas ejecutadas:", ", ".join(real_calls["consultas"]))
print("Run 1 - total ejecuciones:", real_calls["buscar"])

# Run 2: intenta otra búsqueda en el mismo hilo, pero ya no quedan llamadas de tool.
r2 = agent.invoke(
    {
        "messages": [
            HumanMessage(content="Ahora intenta buscar 'PII' y responde en una frase con lo disponible."),
        ]
    },
    config=config,
)

requested_calls_run2 = [m for m in r2["messages"] if hasattr(m, "tool_calls") and m.tool_calls]
requested_total_run2 = sum(len(m.tool_calls) for m in requested_calls_run2)
tool_messages_run2 = [m for m in r2["messages"] if m.__class__.__name__ == "ToolMessage"]
blocked_run2 = [m for m in tool_messages_run2 if "limit" in (m.content or "").lower() or "exceed" in (m.content or "").lower()]
last_ai_run2 = [m for m in r2["messages"] if m.__class__.__name__ == "AIMessage"][-1]
new_exec_run2 = real_calls["buscar"] - 2

print("Run 2 - solicitudes de tool:", requested_total_run2)
print("Run 2 - nuevas ejecuciones:", new_exec_run2)
print("Run 2 - bloqueos por límite:", len(blocked_run2))
print("Run 2 - total acumulado:", real_calls["buscar"])
print("\nRespuesta final (run 2):", (last_ai_run2.content or "")[:250])

Run 1 - consultas ejecutadas: LangChain, middleware
Run 1 - total ejecuciones: 2
Run 2 - solicitudes de tool: 3
Run 2 - nuevas ejecuciones: 0
Run 2 - bloqueos por límite: 1
Run 2 - total acumulado: 2

Respuesta final (run 2): No puedo realizar la búsqueda porque se alcanzó el límite de llamadas a la herramienta; según lo disponible, "PII" (Personally Identifiable Information) es cualquier información que puede identificar a una persona —por ejemplo, nombre, dirección, DNI


Con `exit_behavior="continue"` el agente no falla: deja de ejecutar más herramientas al alcanzar el límite y genera una respuesta final con la información disponible.

---

## 5. ModelFallbackMiddleware: respaldo si el modelo principal falla

**Objetivo:** si la llamada al modelo principal falla (red, cuota, etc.), probar automáticamente con uno o más modelos de respaldo. Se pasan los identificadores en orden de preferencia.

**Importante:** este middleware **no se dispara en ejecución normal**; solo actúa cuando el modelo principal falla. El ejemplo sirve para ver la API; para observar el fallback haría falta simular un fallo del primer modelo.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelFallbackMiddleware
from langchain_core.messages import HumanMessage

agent = create_agent(
    model="gpt-5-mini",
    tools=[],
    middleware=[
        ModelFallbackMiddleware("gpt-5-nano"),
    ],
)
r = agent.invoke({"messages": [HumanMessage(content="Di hola en una palabra.")]})
print("Respuesta:", r["messages"][-1].content)
print("En condiciones normales solo se usa el primero; si fallara, se probaría con gpt-5-nano.")

Respuesta: Hola
En condiciones normales solo se usa el primero; si fallara, se probaría con gpt-5-nano.


---

## Referencia de middleware integrados

| Middleware | Uso típico |
|------------|------------|
| **PIIMiddleware** | Redactar, enmascarar o bloquear PII (email, tarjeta, etc.) en entrada/salida. |
| **SummarizationMiddleware** | Resumir historial cuando se acerca al límite de tokens/mensajes. |
| **ModelCallLimitMiddleware** | Límite de llamadas al modelo por run o por thread (requiere checkpointer). |
| **ToolCallLimitMiddleware** | Límite de llamadas a herramientas; opcionalmente por herramienta. |
| **ModelFallbackMiddleware** | Usar modelos alternativos si el principal falla. |
| **HumanInTheLoopMiddleware** | Pausar para aprobación humana antes de ejecutar ciertas herramientas (requiere checkpointer). |
| **ToolRetryMiddleware** / **ModelRetryMiddleware** | Reintentos con backoff en fallos de herramientas o del modelo. |

Para **memoria a corto plazo** (estado por conversación/hilo) se usa un **checkpointer**, por ejemplo `InMemorySaver()` de `langgraph.checkpoint.memory`. Para **memoria a largo plazo** (persistente entre sesiones) se utilizan backends como **InMemoryStore** o stores persistentes; los middleware que dependen de estado por hilo siguen usando el checkpointer.